# Conversational Clustering — Week 3: Simulated User

**Status:** Week 2 baselines validated. Claude one-shot reaches ARI = 0.36 on topic, 0.07 on methodology — a 0.29 spread to close with the conversational system in Week 4.

**This week's deliverable:** the **simulated user** — an LLM that holds a hidden target labeling and produces natural-language feedback toward it. This is the *measurement infrastructure* of the whole project. Get this wrong and Week 4 measures noise.

**This week we do NOT build:** the conversational system. We're building the user, not the system the user talks to. Week 4 builds the system.

---

## Design choices (decided before coding)

| Choice | Value | Why |
|---|---|---|
| **A. Target visibility** | Description of axis + queryable mapping | Realistic: the user knows what they want at a high level, can verify specific cases on demand |
| **B. View of clustering** | Labels + sizes + 5 sample titles per cluster | Enough to critique, not enough to memorize and copy IDs |
| **C. Personality** | "Astronomer with a clear goal, professional and direct" | Neutral baseline. No personality variance in this study. |
| **D. Stopping** | Fixed budget of 6 turns | Avoids modeling user satisfaction (a separate research question) |
| **E. History** | User sees its own prior messages | More realistic, enables coherence analysis |

**Critical anti-cheating rule:** the simulated user **must not reference paper IDs in its feedback**. It describes intent conceptually ("methodology-based clusters", "split observational from theoretical"), not "move papers 14, 89, 102". If it cheats, we end up measuring the system's ability to follow direct commands, not its ability to infer user intent from natural language — which kills the whole experiment.

---

## Cost forecast

- **Week 3 (build + test):** ~20-30 calls, well under $1.
- **Week 4 (full experiment):** ~800 calls × ~$0.005 with caching = $5-10. Budget realistically $15-20 with iteration.

Sonnet 4.6 (not Haiku) for the simulated user — incoherence at this layer invalidates everything downstream.


## 0. Setup + imports

Reuses the `call_llm()` infrastructure from Week 2 (Anthropic SDK, two-layer caching).


In [ ]:
import os
import json
import time
import hashlib
import re
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
CACHE_DIR = Path("cache_claude")
CACHE_DIR.mkdir(exist_ok=True)
ABSTRACTS_PATH = DATA_DIR / "astro_ph_abstracts.json"

ANTHROPIC_MODEL = "claude-sonnet-4-6"

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"


In [ ]:
# Re-import call_llm from Week 2 logic.
# (In a real project this would be a shared module; for the notebook we copy it.)

def _cache_key(static_block, varying, model):
    h = hashlib.sha256(f"{model}|||{static_block}|||{varying}".encode()).hexdigest()[:16]
    return f"{model}_{h}"

def call_llm(static_block, varying, max_tokens=2048, temperature=0.0, use_cache=True):
    """Provider-agnostic call with prompt + local response caching."""
    model = ANTHROPIC_MODEL
    cache_file = CACHE_DIR / f"{_cache_key(static_block, varying, model)}.json"

    if use_cache and cache_file.exists():
        with open(cache_file) as f:
            payload = json.load(f)
        payload["cached"] = True
        return payload

    import anthropic
    client = anthropic.Anthropic()

    t0 = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temperature,
        system=[{
            "type": "text",
            "text": static_block,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": varying}],
    )
    duration = time.time() - t0

    text = response.content[0].text
    usage = {
        "input_tokens": response.usage.input_tokens,
        "cache_creation_input_tokens": getattr(response.usage, "cache_creation_input_tokens", 0) or 0,
        "cache_read_input_tokens": getattr(response.usage, "cache_read_input_tokens", 0) or 0,
        "output_tokens": response.usage.output_tokens,
    }
    result = {
        "response": text,
        "usage": usage,
        "model": model,
        "cached": False,
        "duration_s": duration,
    }
    if use_cache:
        with open(cache_file, "w") as f:
            json.dump(result, f)
    return result


def estimate_cost(usage):
    """USD cost estimate from usage dict."""
    r_in = 3.0 / 1_000_000   # Sonnet 4.6 input rate per token
    r_out = 15.0 / 1_000_000
    return (
        usage["input_tokens"] * r_in
        + usage["cache_creation_input_tokens"] * r_in * 1.25
        + usage["cache_read_input_tokens"] * r_in * 0.10
        + usage["output_tokens"] * r_out
    )


## 1. Load corpus and Week 2 baselines

We need:
- The 200 abstracts (Week 1)
- One existing clustering to test the simulated user against (Week 2's generic baseline is fine — it's a believable "starting point" for the conversation)
- A target labeling for one hidden axis (we use Claude's detailed-`methodology` baseline as a proxy target for development; in Week 4 you'll hand-label or audit this)


In [ ]:
# Load abstracts
with open(ABSTRACTS_PATH) as f:
    abstracts = json.load(f)
df = pd.DataFrame(abstracts)
print(f"Loaded {len(df)} abstracts")

# Load Week 2 baselines (saved by Week 2 notebook)
with open(DATA_DIR / "claude_baseline_generic.json") as f:
    baseline_generic = json.load(f)
with open(DATA_DIR / "claude_baseline_detailed_methodology.json") as f:
    baseline_methodology = json.load(f)

current_assignments = np.array(baseline_generic["assignments"])
current_labels = {int(k): v for k, v in baseline_generic["labels"].items()}

# Target = methodology axis (the hard one). Treating Claude's methodology baseline
# as the target for *development purposes* — in Week 4 you'll either use this directly
# (after auditing N=30 samples) or hand-label all 200.
target_assignments = np.array(baseline_methodology["assignments"])
target_labels = {int(k): v for k, v in baseline_methodology["labels"].items()}

print(f"\nStarting clustering (generic baseline):")
print(f"  Labels: {list(current_labels.values())}")
print(f"  ARI vs methodology target: {adjusted_rand_score(target_assignments, current_assignments):.3f}")
print(f"\nTarget (methodology):")
print(f"  Labels: {list(target_labels.values())}")


## 2. What the simulated user sees

Per design choice B: labels + sizes + 5 sample titles per cluster. We build a `render_clustering_view()` that produces a clean text summary the user can read and critique.

The view does **not** include paper IDs — only titles. This prevents the user from referencing IDs in feedback (anti-cheating rule).


In [ ]:
def render_clustering_view(
    assignments: np.ndarray,
    labels: dict,
    df: pd.DataFrame,
    samples_per_cluster: int = 5,
    random_state: int = RANDOM_SEED,
) -> str:
    """Produce a text summary of a clustering for the simulated user.

    Critically: no paper IDs in the output, only titles.
    """
    rng = np.random.RandomState(random_state)
    lines = []
    for cid in sorted(set(int(c) for c in assignments if c >= 0)):
        members = np.where(assignments == cid)[0]
        label = labels.get(cid, f"cluster_{cid}")
        lines.append(f"=== Cluster {cid}: \"{label}\" ({len(members)} papers) ===")
        sample_idx = rng.choice(members, size=min(samples_per_cluster, len(members)), replace=False)
        for idx in sorted(sample_idx):
            title = df.iloc[idx]["title"]
            # Truncate very long titles
            if len(title) > 110:
                title = title[:107] + "..."
            lines.append(f"  - {title}")
        lines.append("")
    return "\n".join(lines).strip()


# Smoke test the view
view = render_clustering_view(current_assignments, current_labels, df)
print(view[:1500])
print("...")


## 3. Target description (axis-level, not paper-level)

Per design choice A: the simulated user gets a high-level axis description, not the full mapping. This forces it to give *conceptual* feedback rather than copying IDs.

We reuse the `AXIS_DESCRIPTIONS` from Week 2 — same dictionary, same wording.


In [ ]:
AXIS_DESCRIPTIONS = {
    "topic": '''You want papers clustered by their primary astrophysical subject area:
- Galactic / extragalactic astronomy
- Solar and stellar astrophysics
- Cosmology and large-scale structure
- Earth and planetary astrophysics
- High-energy astrophysics
- Instrumentation and methods''',

    "object_scale": '''You want papers clustered by the PHYSICAL SCALE of their primary object of study:
- Planetary scale (planets, moons, protoplanetary disks)
- Stellar scale (individual stars, binary systems, remnants)
- Galactic scale (single galaxies, galactic structure, star clusters)
- Cosmological scale (galaxy clusters, cosmic web, universe-scale phenomena)''',

    "methodology": '''You want papers clustered by the PRIMARY METHODOLOGY they use, regardless of what they study:
- Observational (telescope/instrument observations, analysis of observational data)
- Theoretical (analytical theory, derivations from first principles)
- Simulation (numerical simulations, N-body, hydrodynamics)
- Instrumental / methods (about an instrument, pipeline, technique itself)''',
}

# For our development scenario we use methodology target
DEVELOPMENT_TARGET_AXIS = "methodology"
print("Target axis for Week 3 development:", DEVELOPMENT_TARGET_AXIS)
print()
print(AXIS_DESCRIPTIONS[DEVELOPMENT_TARGET_AXIS])


## 4. The simulated user

This is the heart of Week 3. The function takes:
- `current_clustering_view` (string) — what the user sees
- `target_description` (string) — the axis description they internalize as their goal
- `history` (list of messages) — past turns of this conversation
- `turn_idx` (int) — which turn we're on

And returns a natural-language feedback message.

**The prompt is designed to enforce three properties:**

1. **No paper IDs in feedback.** Explicit instruction + the user can't see IDs in the view anyway.
2. **Conceptual feedback.** "These clusters mix observational and simulation work" not "move these to that".
3. **Stable goal across turns.** Anchored in the target description that stays constant.

The prompt also gives the user a *persona*: an astronomer with a clear methodology-clustering goal in mind. We avoid extreme personalities (frustrated, super polite, etc.) — we want a neutral baseline.


In [ ]:
SIMULATED_USER_SYSTEM_PROMPT = '''You are simulating an astronomer using an automated clustering tool to organize a collection of astro-ph paper abstracts. You have a specific clustering goal in mind:

{target_description}

The current clustering does not necessarily match your goal. Your job is to give the system natural-language feedback so that it can update the clustering. You will be shown the current clusters (labels, sizes, sample titles) and asked to give feedback for one turn.

CRITICAL RULES for your feedback:

1. **Do NOT reference paper IDs.** You cannot see IDs anyway. Describe what you want in terms of concepts ("the simulation papers", "the observational work"), not specific items.

2. **Be conceptual, not list-based.** Don't enumerate every paper that's misplaced. Describe the pattern: "Cluster 2 seems to mix observational and theoretical papers — those should be separated by methodology."

3. **Stay anchored to your goal.** Your goal (above) doesn't change across turns. If a previous turn's feedback was not fully addressed, you may restate it more clearly, but do not contradict yourself.

4. **Be specific enough to be actionable.** "I don't like these clusters" is useless. "The 'High-Energy Astrophysics' cluster mixes simulation papers with observational ones — split by methodology" is actionable.

5. **Be concise.** 2-4 sentences per turn. Real users don't write essays.

6. **Don't propose specific operations.** Don't say "merge clusters 1 and 4" or "increase K to 7". Describe the *outcome* you want, not the *operation*. The system decides which operations to apply.

7. **Don't claim satisfaction unless the clustering genuinely matches your goal.** If you're being asked to give feedback, there's still a gap.

Respond ONLY with the feedback message — no preamble, no meta-commentary, no quote marks. Just what you would say.'''


SIMULATED_USER_TURN_PROMPT = '''Turn {turn_idx} of {max_turns}.

Current clustering:

{clustering_view}

{history_section}Give your next piece of feedback. Remember: conceptual, concise, no paper IDs, anchored to your goal.'''


def format_history_for_user(history: list[dict]) -> str:
    """Format past turns for the user to see their own prior messages."""
    if not history:
        return ""
    lines = ["Your previous feedback in this conversation:\n"]
    for turn in history:
        lines.append(f"  Turn {turn['turn_idx']}: \"{turn['feedback']}\"")
    lines.append("")
    return "\n".join(lines) + "\n"


def simulated_user_feedback(
    current_clustering_view: str,
    target_description: str,
    history: list[dict],
    turn_idx: int,
    max_turns: int = 6,
    use_cache: bool = True,
) -> dict:
    """Ask the simulated user for one turn of feedback.

    Returns a dict with the feedback text and call metadata.
    """
    static = SIMULATED_USER_SYSTEM_PROMPT.format(target_description=target_description)
    varying = SIMULATED_USER_TURN_PROMPT.format(
        turn_idx=turn_idx,
        max_turns=max_turns,
        clustering_view=current_clustering_view,
        history_section=format_history_for_user(history),
    )

    # Note: we use slightly higher temperature here than for clustering baselines —
    # we want some natural variation in how the user phrases things, like a real
    # user would. But not so high that the goal becomes incoherent.
    llm_out = call_llm(static, varying, max_tokens=512, temperature=0.4, use_cache=use_cache)

    return {
        "feedback": llm_out["response"].strip(),
        "turn_idx": turn_idx,
        "usage": llm_out["usage"],
        "cost_usd": estimate_cost(llm_out["usage"]) if not llm_out["cached"] else 0.0,
        "duration_s": llm_out["duration_s"],
        "cached": llm_out["cached"],
    }


## 5. Smoke test — single turn

Run the simulated user once against the generic-baseline clustering, with methodology as the target. We're looking for:

- Feedback is conceptual, no paper IDs
- Feedback references methodology concepts (observational, simulation, theoretical, instrumental)
- Length is reasonable (2-4 sentences)
- Tone matches "professional astronomer"


In [ ]:
result = simulated_user_feedback(
    current_clustering_view=render_clustering_view(current_assignments, current_labels, df),
    target_description=AXIS_DESCRIPTIONS[DEVELOPMENT_TARGET_AXIS],
    history=[],
    turn_idx=1,
    max_turns=6,
)

print(f"Feedback:\n  {result['feedback']}")
print(f"\nCost: ${result['cost_usd']:.5f}  Duration: {result['duration_s']:.1f}s  Cached: {result['cached']}")


In [ ]:
# Anti-cheating check: does the feedback reference paper IDs?
feedback = result["feedback"]
id_pattern = re.compile(r"\b(?:paper|abstract|item)\s*[#]?\s*\d+\b|\[\s*\d+\s*\]", re.IGNORECASE)
matches = id_pattern.findall(feedback)
if matches:
    print(f"⚠️  POTENTIAL CHEATING — paper IDs found in feedback: {matches}")
    print("Look at the feedback above. If the IDs are real references, the prompt needs tightening.")
else:
    print("✓ No paper IDs in feedback")

# Methodology-keyword check: does the feedback engage with the methodology axis?
method_kw = ["observation", "theoretical", "theory", "simulation", "instrument", "method"]
hits = [kw for kw in method_kw if kw in feedback.lower()]
if hits:
    print(f"✓ Methodology concepts engaged: {hits}")
else:
    print(f"⚠️  No methodology keywords in feedback — is the user on-target?")


## 6. Multi-turn dry-run

Now we simulate a 4-turn conversation. **Important caveat:** since we haven't built the conversational system yet (that's Week 4), the clustering does NOT update between turns. The simulated user is shown the same view turn after turn.

This is artificial but informative. What we want to verify:

1. **Coherence across turns.** Does the user keep pushing toward the same goal, or does it drift?
2. **No paper ID leakage** at any turn (even when frustrated).
3. **Natural escalation.** A real user would say roughly the same thing more emphatically when ignored. Does ours?
4. **Cost per turn** is stable and predictable.

If the user changes its mind between turns, or starts citing specific paper IDs, the prompt needs hardening before Week 4.


In [ ]:
# Dry-run: 4 turns, clustering stays the same (no system in the loop yet)

view = render_clustering_view(current_assignments, current_labels, df)
target_desc = AXIS_DESCRIPTIONS[DEVELOPMENT_TARGET_AXIS]
history = []
total_cost = 0.0

for turn in range(1, 5):
    print(f"\n{'='*60}\nTURN {turn}\n{'='*60}")
    result = simulated_user_feedback(
        current_clustering_view=view,
        target_description=target_desc,
        history=history,
        turn_idx=turn,
        max_turns=6,
    )
    print(f"User: {result['feedback']}")
    print(f"\n  [cost: ${result['cost_usd']:.5f}  duration: {result['duration_s']:.1f}s  cached: {result['cached']}]")
    history.append({"turn_idx": turn, "feedback": result["feedback"]})
    total_cost += result["cost_usd"]

print(f"\n\nTotal cost for 4-turn dry-run: ${total_cost:.4f}")


## 7. Quality audit on the dry-run

Programmatic checks across all 4 turns:

1. **No paper IDs anywhere.**
2. **Each turn engages with methodology concepts.**
3. **Goal consistency** — measured loosely by checking that key methodology terms appear consistently across turns.


In [ ]:
print("Audit across all turns:\n")

id_pattern = re.compile(r"\b(?:paper|abstract|item)\s*[#]?\s*\d+\b|\[\s*\d+\s*\]", re.IGNORECASE)
method_kw = ["observation", "theoretical", "theory", "simulation", "instrument", "method"]

cheating_turns = []
off_target_turns = []
for turn in history:
    fb = turn["feedback"]
    has_id = bool(id_pattern.findall(fb))
    has_method = any(kw in fb.lower() for kw in method_kw)
    if has_id:
        cheating_turns.append(turn["turn_idx"])
    if not has_method:
        off_target_turns.append(turn["turn_idx"])

if cheating_turns:
    print(f"⚠️  Paper IDs found in turns: {cheating_turns}")
else:
    print(f"✓ No paper IDs in any turn ({len(history)} checked)")

if off_target_turns:
    print(f"⚠️  No methodology keywords in turns: {off_target_turns}")
else:
    print(f"✓ Every turn engaged methodology concepts")

# Word-overlap consistency check (informal)
print("\nKey concept frequency across turns:")
for kw in method_kw:
    counts = sum(1 for t in history if kw in t["feedback"].lower())
    if counts > 0:
        print(f"  '{kw}': appears in {counts}/{len(history)} turns")


## 8. Sanity test — different target produces different feedback

If we swap the target from `methodology` to `object_scale`, the feedback should change substantively. If it doesn't, the user isn't actually reading the target description.


In [ ]:
result_alt = simulated_user_feedback(
    current_clustering_view=view,
    target_description=AXIS_DESCRIPTIONS["object_scale"],
    history=[],
    turn_idx=1,
    max_turns=6,
)
print(f"With object_scale target, turn 1:\n  {result_alt['feedback']}\n")

# Check that this feedback engages scale concepts
scale_kw = ["planet", "stellar", "star", "galactic", "galaxy", "cosmolog", "scale"]
scale_hits = [kw for kw in scale_kw if kw in result_alt["feedback"].lower()]
method_hits = [kw for kw in method_kw if kw in result_alt["feedback"].lower()]
print(f"Scale keywords hit:       {scale_hits}")
print(f"Methodology keywords hit: {method_hits}")
print()
if scale_hits and not method_hits:
    print("✓ Target swap produces target-appropriate feedback")
elif method_hits and not scale_hits:
    print("⚠️  User is stuck on methodology — not reading the target description")
else:
    print("• Mixed signal — inspect manually")


## 9. Persist the simulated-user infrastructure

Save:
- The dry-run conversation history (useful for Week 4 baseline comparisons)
- The simulated-user prompts (versioned — these *will* evolve and you want to track which version produced which results)


In [ ]:
# Save the dry-run history as a baseline artifact
dry_run = {
    "target_axis": DEVELOPMENT_TARGET_AXIS,
    "starting_clustering": "generic",
    "n_turns": len(history),
    "history": history,
    "total_cost_usd": total_cost,
    "notes": "Dry-run with no system in the loop — clustering does not update between turns.",
}
out = DATA_DIR / "week3_simulated_user_dryrun.json"
with open(out, "w") as f:
    json.dump(dry_run, f, indent=2)
print(f"Saved dry-run to {out}")

# Save prompt versions
prompts = {
    "version": "v0.1",
    "date": time.strftime("%Y-%m-%d"),
    "system_prompt": SIMULATED_USER_SYSTEM_PROMPT,
    "turn_prompt": SIMULATED_USER_TURN_PROMPT,
    "design_choices": {
        "target_visibility": "axis-description only (no paper-level mapping)",
        "clustering_view": "labels + sizes + 5 sample titles, no IDs",
        "personality": "professional astronomer with clear goal",
        "stopping": "fixed 6-turn budget",
        "history": "user sees own past messages",
        "temperature": 0.4,
    },
}
out = DATA_DIR / "week3_simulated_user_prompts_v01.json"
with open(out, "w") as f:
    json.dump(prompts, f, indent=2)
print(f"Saved prompt v0.1 to {out}")


## Week 3 checklist

- [ ] Single-turn smoke test produces conceptual, methodology-aware feedback
- [ ] 4-turn dry-run shows no paper ID leakage anywhere
- [ ] Every dry-run turn engages methodology keywords
- [ ] Target-swap test produces clearly different feedback (object_scale vs methodology)
- [ ] Total Week 3 cost under $1
- [ ] Saved dry-run + prompt v0.1 to disk

## What you may need to fix

**ID leakage.** If the user references paper IDs despite the prompt forbidding it, you have two options:
- Strengthen the system prompt (add explicit examples of forbidden language).
- Post-process the output to strip ID references before they reach the system. Heavy-handed but reliable.

**Goal drift across turns.** If turn 3 is about object scale and turn 4 is about something else, the user isn't anchoring. Try:
- Lower temperature (0.2 instead of 0.4).
- Restate the goal at every turn instead of relying on the system prompt sticking.

**Same feedback every turn.** If the user just repeats the same critique verbatim, that's OK for a dry-run (clustering didn't change) but might be a problem in Week 4 if the clustering DOES update but the user doesn't notice the change. Watch for this in Week 4.

**Target swap produces same feedback.** Critical failure — the user isn't reading the target. Inspect the prompt for placeholder mistakes.

## Things to write in `notes.md`

- Total Week 3 cost (number).
- Prompt version locked at the end of the week (v0.1, or v0.2 if you iterated).
- Anything notable in the dry-run conversation — coherence, naturalness, surprises.
- For Week 4: which simulated-user version (v0.1, etc.) gets used? Lock it before launching experiments.

## Week 4 preview

Now we build the system the simulated user talks to. The system reads the feedback and emits *operations* on the clustering pipeline (split, merge, change_k, etc.). One turn of the loop becomes:

1. System shows current clustering
2. Simulated user gives feedback (← what we built this week)
3. **System parses feedback → operations → new clustering** (← Week 4)
4. ARI logged

Then we run this loop many times across (3 targets × 3 systems × N runs) and analyze.
